# Практическая работа №6

## Сопровождение объектов

## Задание

### Цель

Знакомство с методами сопровождения объектов, формирование навыков реализации методов сопровождения объектов на языке Python.

### Задачи

Выполнение практической работы направлено на исследование методов сопровождения объектов

## Ход работы

Для работы выбран [набор данных](https://www.kaggle.com/datasets/kmader/videoobjecttracking/data), посвященный задаче сопровождения объектов.

Из набора выбран один из экземпляров данных, где нужно произвести сопровождение коробки сока на последовательности изображений. Для использованного набора доступен файл с разметкой, который содержит координаты bounding box сопровождаемой коробки для каждого кадра. Разметка будет использована для вычисления метрик качества для каждого метода сопровождения объектов.

### Импорты и вспомогательные функции

### Подготовка окружения

Проверяю наличие зависимостей для примеров ниже.


In [ ]:
%pip install numpy matplotlib opencv-contrib-python pandas tqdm pillow


In [ ]:
import os
from collections import defaultdict

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython import display
from tqdm import tqdm

DATA_DIR = 'data'

В качестве метрики качества используется IoU

In [ ]:
def iou_score(box_true, box_pred):
    x1t, y1t, x2t, y2t = box_true
    x1p, y1p, x2p, y2p = box_pred
    x_left = max(x1t, x1p)
    y_top = max(y1t, y1p)
    x_right = min(x2t, x2p)
    y_bottom = min(y2t, y2p)
    
    if x_right < x_left or y_bottom < y_top:
        return 0
    
    intersection = (x_right - x_left) * (y_bottom - y_top)
    box_true_area = (x2t - x1t) * (y2t - y1t)
    box_pred_area = (x2p - x1p) * (y2p - y1p)
    
    return intersection / (box_true_area + box_pred_area - intersection)

### Выполнение сопровождения объектов

In [ ]:
gt = pd.read_table(os.path.join(DATA_DIR, 'gt.txt'), header=None, sep=',', names=['x0', 'y0', 'x1', 'y1'])
img_names = sorted([filename for filename in os.listdir(DATA_DIR) if filename.endswith('.jpg')])
if len(img_names) != len(gt):
    min_len = min(len(img_names), len(gt))
    if len(img_names) > min_len:
        img_names = img_names[:min_len]
    if len(gt) > min_len:
        gt = gt.iloc[:min_len].reset_index(drop=True)
    print(f'Количество кадров ({len(img_names)}) не совпало с аннотациями ({len(gt)}). Использую общую часть.')

init_image = cv2.imread(os.path.join(DATA_DIR, img_names[0]))
init_markup = gt.iloc[0]
init_bbox = (int(init_markup['x0']), int(init_markup['y0']), int(init_markup['x1']) - int(init_markup['x0']), int(init_markup['y1']) - int(init_markup['y0']))
image_height, image_width, _ = init_image.shape

demo_image = init_image.copy()
cv2.rectangle(demo_image, init_bbox, (0, 255, 0), 1)
plt.imshow(demo_image);


In [ ]:
metrics = defaultdict(list)
video_writers = {}

trackers = {
    'CSRT': cv2.TrackerCSRT_create(),
    'BOOSTING': cv2.legacy.TrackerBoosting_create(),
    'MIL': cv2.TrackerMIL_create(),
    'KCF': cv2.TrackerKCF_create(),
    'TLD': cv2.legacy.TrackerTLD_create(),
    'MEDIANFLOW': cv2.legacy.TrackerMedianFlow_create(),
    'MOSSE': cv2.legacy.TrackerMOSSE_create()
}

In [ ]:
for name, tracker in trackers.items():
    tracker.init(init_image, init_bbox)
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writers[name] = cv2.VideoWriter(f'{name}.mp4', fourcc, 15.0, (image_width, image_height))

for i, img_name in tqdm(enumerate(img_names), total=len(img_names), desc='Трекинг объекта'):
    frame = cv2.imread(os.path.join(DATA_DIR, img_name))
    markup = gt.iloc[i]
    gt_bbox = (int(markup['x0']), int(markup['y0']), int(markup['x1']) - int(markup['x0']), int(markup['y1']) - int(markup['y0']))
    
    for name, tracker in trackers.items():
        tracker_frame = frame.copy()
        ok, bbox = tracker.update(tracker_frame)
        if ok:
            x, y, w, h = (int(val) for val in bbox)
            cv2.rectangle(tracker_frame, (x, y), (x+w, y+h), (0, 0, 255), 2)
            
            gt_box = (int(markup['x0']), int(markup['y0']), int(markup['x1']), int(markup['y1']))
            tracker_box = (x, y, x+w, y+h)
            metrics[name].append(iou_score(tracker_box, gt_box))
        else:
            metrics[name].append(0)
            cv2.putText(tracker_frame, f'{name} Lost', (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
        
        cv2.rectangle(tracker_frame, gt_bbox, (0, 128, 0), 1)
        video_writers[name].write(tracker_frame)

for writer in video_writers.values():
    writer.release()

В результате сопровождения рассчитывается средний IoU

In [ ]:
average_iou = {}
for tracker_name in [*trackers]:
    print(f'{tracker_name}: {np.mean(metrics[tracker_name]):.4f}')

### CSRT

Получен хороший общий результат, но заметны колебания вертикальных границ bounding box. При отдалении объекта метод адаптировал размер bounding box. С небольшим разрывом занял второе место по точности.

### Boosting

Очень хороший результат сопровождения, колебаний центра и границ bounding box не обнаружено. Среднее значение метрики качества обусловлено тем, что метод не адаптировал размер bounding box при отдалении объекта.

### MIL

В процессе сопровождения заметно смещение центра bounding box относительно разметки. Также метод не адаптирует размер рамки при отдалении объекта.

### KCF

Очень хорошая точность сопровождения, на уровне метода Boosting. Данный метод также не адаптирует размер bounding box при отдалении объекта. В процессе сопровождения объект был на некоторое время потерян (затем снова удалось его найти), что сказалось на значении метрики качества.

### TLD

Из всех методов, не считая MOSSE, этот демонстрирует самый неудовлетворительный результат. На протяжении всего процесса сопровождения координаты центра bounding box смещены относительно разметки. Также заметны колебания границ при изменении направления движения камеры. Полученное значение метрики качества (третье сверху) объясняется тем, что метод адаптирует размер bounding box при отдалении объекта. Если бы размеры объекта оставались неизменными, то данный метод занял бы место ближе к концу рейтинга.

### MedianFlow

Результат сопровождения практически полностью совпадает с разметкой. При отдалении объекта метод адаптировал размер bounding box, что позволило добиться высокого значения метрики качества. Лучшая точность работы среди рассмотренных методов.

### MOSSE

В начале сопровождения потерял объект и больше не смог его обнаружить. По этой причине метод показал худший результат среди всех рассмотренных.

## Вывод

Были рассмотрены различные методы сопровождения объектов, и изучены способы их применения.

Из рассмотренных методов лучше всего себя показал MedianFlow.